# 11 — Elyra：GPU 推論

**Pipeline Editor** 第 2 步。必須使用含 **Modulus + CUDA** 的 Runtime Image（CorrDiff Workbench 映像），並配置 **GPU=1**。

與 Phase 1 Job / `01-test-corrdiff-inference.ipynb` 相同腳本：`bin/inference_v1.py`。


In [ ]:
import os
import sys
import subprocess
from pathlib import Path

SHOME = os.environ.get("SHOME", "/mnt/corrdiff")
DATE = os.environ.get("DATE", "20260707")
FC = os.environ.get("FC_VERSION", "ec46day")
OP = os.environ.get("OP_VERSION", "opv1")
CD = os.environ.get("CORRDIFF_VERSION", "corrdiff_v1")
CONFIG = f"{SHOME}/workdir/{DATE}/config.yaml"
OUTPUT_NC = f"{SHOME}/dtg/EC_S2S_AIPP/{DATE}/CorrdiffOutput_EC_RAW_{DATE}.nc"

os.environ["HOME"] = "/tmp"
os.environ["LOCAL_CACHE"] = "/tmp/.cache/modulus"
os.makedirs(os.environ["LOCAL_CACHE"], exist_ok=True)

import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("CUDA not available — set Elyra node GPU=1 and CorrDiff GPU runtime image")

import modulus
print("modulus OK")
assert Path(CONFIG).is_file(), f"Missing config: {CONFIG} (run 10-elyra-smoke first)"
print(f"CONFIG={CONFIG}")


In [ ]:
os.makedirs(f"{SHOME}/dtg/EC_S2S_AIPP/{DATE}", exist_ok=True)
env = os.environ.copy()
env["PYTHONPATH"] = f"{SHOME}/bin:" + env.get("PYTHONPATH", "")
env["HOME"] = "/tmp"
env["LOCAL_CACHE"] = "/tmp/.cache/modulus"

cmd = [
    sys.executable, f"{SHOME}/bin/inference_v1.py",
    DATE, CONFIG, FC, OP, CD,
]
print(" ".join(cmd))
proc = subprocess.run(cmd, cwd=f"{SHOME}/bin", env=env)
print("exit code:", proc.returncode)
assert proc.returncode == 0, "inference_v1.py failed"
assert Path(OUTPUT_NC).is_file(), f"Missing output: {OUTPUT_NC}"
print(f"[OK] {OUTPUT_NC} ({Path(OUTPUT_NC).stat().st_size} bytes)")
print("Inference completed successfully.")
